# Tuchanka design notebook

Design, preview, and export gcode for the Tuchanka (Prusa XL 5-tool) printer.

Run cells top to bottom with shift+enter. If you change a cell, re-run it and all cells below it.

In [ ]:
import fullcontrol as fc
from fullcontrol.devices.personal.tuchanka.setup import starting_steps, ending_steps, base_settings, load_profile, next_output_path

In [15]:
# printer / startup parameters

design_name  = 'xl_fc_test1'
initial_tool = 3
tools_used   = [3]
print_area   = (20, 20, 180, 180)  # (x_min, y_min, x_max, y_max) mm

profile = load_profile('jessie_silky_pla_0.8')  # loads profiles/petg.json — swap name to change material

startup = starting_steps(
    initial_tool=initial_tool,
    tools_used=tools_used,
    first_layer_temps=profile['first_layer_temps'],
    idle_temps=profile['idle_temps'],
    mbl_temp=profile['mbl_temp'],
    bed_temp=profile['bed_temp'],
    travel_speed=profile['travel_speed'],
    zhop=profile['zhop'],
    print_area=print_area,
    heat_soak=False,
    nozzle_clean=False,
)

In [16]:
# design parameters — pulled from profile, override here if needed

EW         = profile['extrusion_width']   # mm
EH         = profile['extrusion_height']  # mm
initial_z  = EH * 0.6
print_speed = profile['print_speed']      # mm/min

In [17]:
# H design — single layer, outline only, centered on the XL bed (360x360mm)
#
# Letter dimensions (all mm):
#   height:       100
#   total width:   80  (left bar 15 | gap 50 | right bar 15)
#   crossbar height: 15, centered vertically
#
# Bed center: (180, 180)

letter_h = 100
letter_w = 80
stroke_w = 15
cx, cy   = 180.0, 180.0

x0 = cx - letter_w / 2        # 140
x1 = x0 + stroke_w             # 155
x2 = cx + letter_w / 2 - stroke_w  # 205
x3 = cx + letter_w / 2        # 220

y_bot  = cy - letter_h / 2    # 130
y_top  = cy + letter_h / 2    # 230
cb_bot = cy - stroke_w / 2    # 172.5
cb_top = cy + stroke_w / 2    # 187.5

z = initial_z

# Trace the H outline as a single continuous path with travel moves between segments.
# Left bar → crossbar → right bar, avoiding backtracking.
design_steps = [
    # --- left bar ---
    fc.Extruder(on=False),
    fc.Point(x=x0, y=y_bot, z=z),
    fc.Extruder(on=True),
    fc.Point(x=x0, y=y_top),
    fc.Point(x=x1, y=y_top),
    fc.Point(x=x1, y=cb_top),
    # --- crossbar (left to right along top edge) ---
    fc.Point(x=x2, y=cb_top),
    # --- right bar (up to top) ---
    fc.Point(x=x2, y=y_top),
    fc.Point(x=x3, y=y_top),
    fc.Point(x=x3, y=y_bot),
    fc.Point(x=x2, y=y_bot),
    fc.Point(x=x2, y=cb_bot),
    # --- crossbar (right to left along bottom edge) ---
    fc.Point(x=x1, y=cb_bot),
    # --- left bar (down to bottom) ---
    fc.Point(x=x1, y=y_bot),
    fc.Point(x=x0, y=y_bot),
]

In [18]:
# preview the design (startup gcode steps are excluded — they're not geometry)

fc.transform(design_steps, 'plot', fc.PlotControls(style='line', zoom=0.7))

# for a more realistic preview showing actual extrusion widths, uncomment:
# fc.transform(design_steps, 'plot', fc.PlotControls(style='tube', zoom=0.7, initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))

In [ ]:
# generate and save gcode

shutdown = ending_steps(travel_speed=profile['travel_speed'], present_print=True)
steps = startup + design_steps + shutdown

gcode_controls = fc.GcodeControls(
    printer_name='custom',
    save_as=next_output_path(design_name, output_dir='personal_models/gcode'),
    include_date=False,
    initialization_data=base_settings(
        print_speed=print_speed,
        extrusion_width=EW,
        extrusion_height=EH,
        nozzle_temp=profile['first_layer_temps'][initial_tool],
        bed_temp=profile['bed_temp'],
    ),
)
gcode = fc.transform(steps, 'gcode', gcode_controls)

---
*Tuchanka notebook — Petricore Labs*